# 🎬 YouTube Automation Tool

1. Run **Cell 1** — installs everything (~2 min, once)
2. Run **Cell 2** — opens the web UI, copy the `gradio.live` link to your phone

In [ ]:
# ── CELL 1: Install ───────────────────────────────────────────────────────────
print('Installing packages... (~2 minutes)')
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'edge-tts', 'moviepy==1.0.3', 'Pillow', 'requests',
                'imageio==2.31.6', 'imageio-ffmpeg', 'gradio', 'nest-asyncio',
                '-q'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg',
                'fonts-dejavu-core'], capture_output=True)
print('✅ All installed! Run Cell 2.')

In [ ]:
# ── CELL 2: Web interface ─────────────────────────────────────────────────────
import asyncio, shutil, requests, nest_asyncio, gradio as gr
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import edge_tts

nest_asyncio.apply()  # fixes asyncio conflict inside Colab

VOICES = {
    "Narrator Male (US) — Christopher":  "en-US-ChristopherNeural",
    "Narrator Female (US) — Jenny":      "en-US-JennyNeural",
    "Male (US) — Guy":                   "en-US-GuyNeural",
    "Female (US) — Aria":                "en-US-AriaNeural",
    "Male (UK) — Ryan":                  "en-GB-RyanNeural",
    "Female (UK) — Sonia":               "en-GB-SoniaNeural",
}

# ── TTS ───────────────────────────────────────────────────────────────────────
async def _tts_async(text, voice, path):
    words = []
    comm = edge_tts.Communicate(text, voice)
    with open(path, 'wb') as f:
        async for chunk in comm.stream():
            if chunk['type'] == 'audio':
                f.write(chunk['data'])
            elif chunk['type'] == 'WordBoundary':
                words.append({
                    'word':     chunk['text'],
                    'start':    chunk['offset']   / 10_000_000,
                    'duration': chunk['duration'] / 10_000_000,
                })
    return words

def run_tts(text, voice, path):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(_tts_async(text, voice, path))

# ── B-roll ────────────────────────────────────────────────────────────────────
def fetch_broll(topic, key, tmp, count=6):
    keywords = [w.lower() for w in topic.split() if len(w) > 3][:3]
    clips = []
    for i, kw in enumerate(keywords):
        try:
            r = requests.get(
                'https://api.pexels.com/videos/search',
                headers={'Authorization': key},
                params={'query': kw, 'per_page': 2, 'orientation': 'landscape'},
                timeout=15
            )
            if r.status_code != 200:
                continue
            for video in r.json().get('videos', []):
                files = video.get('video_files', [])
                hd = [f for f in files if f.get('quality') == 'hd' and f.get('width', 0) >= 1280]
                link = (hd or files or [None])[0]
                if not link:
                    continue
                p = tmp / f'broll_{i}_{len(clips)}.mp4'
                resp = requests.get(link['link'], stream=True, timeout=60)
                with open(p, 'wb') as f:
                    for chunk in resp.iter_content(8192):
                        f.write(chunk)
                clips.append(p)
                if len(clips) >= count:
                    return clips
        except Exception as e:
            print(f'B-roll fetch error: {e}')
    return clips

# ── Video assembly ────────────────────────────────────────────────────────────
def make_video(broll, audio_path, words, out_path):
    from moviepy.editor import (
        VideoFileClip, AudioFileClip, ColorClip, TextClip,
        concatenate_videoclips, CompositeVideoClip
    )
    SIZE = (1920, 1080)
    audio = AudioFileClip(str(audio_path))
    total = audio.duration

    # Build background
    bg_clips, cur, idx = [], 0.0, 0
    if broll:
        while cur < total and idx < len(broll) * 3:
            p = broll[idx % len(broll)]
            try:
                c = VideoFileClip(str(p)).without_audio().resize(SIZE)
                rem = total - cur
                if c.duration > rem:
                    c = c.subclip(0, rem)
                bg_clips.append(c.set_start(cur))
                cur += c.duration
            except:
                pass
            idx += 1

    if not bg_clips:
        bg_clips = [ColorClip(SIZE, color=(10, 10, 25), duration=total)]

    bg = concatenate_videoclips(bg_clips, method='compose').set_audio(audio)

    # Subtitles
    sub_clips = []
    groups, group = [], []
    for w in words:
        group.append(w)
        if len(group) == 7:
            groups.append(group)
            group = []
    if group:
        groups.append(group)

    for g in groups:
        text  = ' '.join(w['word'] for w in g)
        start = g[0]['start']
        end   = min(g[-1]['start'] + g[-1]['duration'], total)
        dur   = end - start
        if dur <= 0:
            continue
        try:
            tc = (TextClip(text, fontsize=56, color='white',
                           font='DejaVu-Sans-Bold',
                           stroke_color='black', stroke_width=2,
                           method='caption', size=(1600, None))
                  .set_start(start).set_duration(dur)
                  .set_position(('center', 880)))
            sub_clips.append(tc)
        except Exception as e:
            print(f'Subtitle error: {e}')

    final = CompositeVideoClip([bg] + sub_clips, size=SIZE)
    final.write_videofile(str(out_path), fps=30, codec='libx264',
                          audio_codec='aac', preset='medium',
                          threads=2, logger=None)
    audio.close()

# ── Thumbnail ─────────────────────────────────────────────────────────────────
def make_thumbnail(title, out_path):
    SIZE = (1280, 720)
    img  = Image.new('RGB', SIZE, (15, 15, 35))
    draw = ImageDraw.Draw(img)
    # dark gradient bottom
    for y in range(SIZE[1] // 2, SIZE[1]):
        draw.line([(0, y), (SIZE[0], y)], fill=(0, 0, 0))
    # font
    font_candidates = [
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
        '/usr/share/fonts/truetype/freefont/FreeSansBold.ttf',
    ]
    font = ImageFont.load_default()
    for fp in font_candidates:
        try:
            font = ImageFont.truetype(fp, 72)
            break
        except:
            pass
    # wrap text
    words_list = title.upper().split()
    lines, line = [], []
    for w in words_list:
        line.append(w)
        if draw.textbbox((0, 0), ' '.join(line), font=font)[2] > SIZE[0] - 100 and len(line) > 1:
            line.pop()
            lines.append(' '.join(line))
            line = [w]
    if line:
        lines.append(' '.join(line))
    # draw
    y = (SIZE[1] - len(lines) * 82) // 2 + 80
    for text in lines:
        bb = draw.textbbox((0, 0), text, font=font)
        x  = (SIZE[0] - (bb[2] - bb[0])) // 2
        draw.text((x + 3, y + 3), text, fill=(0, 0, 0), font=font)
        draw.text((x, y),         text, fill=(255, 220, 50), font=font)
        y += 82
    img.save(str(out_path), 'JPEG', quality=95)

# ── Pipeline ──────────────────────────────────────────────────────────────────
def generate(topic, script, voice_label, pexels_key, progress=gr.Progress()):
    if not topic.strip():
        raise gr.Error('Please enter a topic')
    if not script.strip():
        raise gr.Error('Please enter a script')

    voice = VOICES[voice_label]
    safe  = ''.join(c if c.isalnum() else '_' for c in topic)[:35].lower()
    tmp   = Path('/content/tmp')
    tmp.mkdir(exist_ok=True)
    out_mp4   = Path(f'/content/{safe}.mp4')
    out_thumb = Path(f'/content/{safe}_thumbnail.jpg')

    progress(0.1, desc='Generating voiceover...')
    words = run_tts(script, voice, tmp / 'audio.mp3')
    print(f'TTS: {len(words)} words')

    broll = []
    key = pexels_key.strip() if pexels_key else ''
    if key:
        progress(0.3, desc='Fetching B-roll...')
        broll = fetch_broll(topic, key, tmp)
        print(f'B-roll: {len(broll)} clips')
    else:
        print('No Pexels key — using dark background')

    progress(0.5, desc='Assembling video (3-5 min)...')
    make_video(broll, tmp / 'audio.mp3', words, out_mp4)

    progress(0.9, desc='Creating thumbnail...')
    make_thumbnail(topic, out_thumb)

    shutil.rmtree(tmp, ignore_errors=True)
    progress(1.0, desc='Done!')
    return str(out_mp4), str(out_thumb)

# ── UI ────────────────────────────────────────────────────────────────────────
with gr.Blocks(title='YouTube Automation', theme=gr.themes.Soft()) as app:
    gr.Markdown('# 🎬 YouTube Automation Tool')
    gr.Markdown('Enter your topic and script, then press **Generate Video**.')

    with gr.Row():
        with gr.Column():
            inp_topic  = gr.Textbox(label='📌 Topic',
                                    placeholder='e.g. The Fall of the Roman Empire')
            inp_script = gr.Textbox(label='📝 Script',
                                    placeholder='Paste your full script here...',
                                    lines=10)
            inp_voice  = gr.Dropdown(label='🎙 Voice',
                                     choices=list(VOICES.keys()),
                                     value=list(VOICES.keys())[0])
            inp_pexels = gr.Textbox(label='🎥 Pexels API Key (optional — for video backgrounds)',
                                    placeholder='Get free key at pexels.com/api',
                                    type='password')
            btn = gr.Button('🚀 Generate Video', variant='primary', size='lg')

        with gr.Column():
            out_video = gr.Video(label='📹 Result Video')
            out_thumb = gr.Image(label='🖼 Thumbnail')

    btn.click(
        fn=generate,
        inputs=[inp_topic, inp_script, inp_voice, inp_pexels],
        outputs=[out_video, out_thumb],
    )

app.launch(share=True)